# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghayoorahmed7/flyrank_ml_intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Target: ctr (regression), not the Week-4 rule's action flag.

My Week-4 baseline's action label (refresh / no_action) is defined by a rule I wrote
(freshness_tier in a stale bucket AND underperform_ratio < 0.5 AND visible AND has position
data). A label that comes from someone's rule means a model trained on it just learns the rule,
not the world. So instead I predict ctr itself — an observed number, not something I defined.

This also gives a fair comparison to Week 4: my baseline rule's own expected_ctr_for_tier step
(mean CTR per position_tier) is already a CTR predictor — that tier-mean is my baseline model.
Week 5's job is to see whether a real regression model, using more signal (content type, word
count, competition, freshness, search volume), can beat that mean-CTR baseline on held-out data,
using the same metric (MAE).

Method: Gradient Boosting Regressor and Random Forest — CTR is noisy and likely non-linear in
position/competition/freshness interactions, which tree ensembles handle better than a linear
model.

Leakage guardrails: excluded clicks_90d (used to compute ctr), everything downstream of a click
(engagement_rate, scroll_rate, ai_traffic_pct, pageviews_90d, sessions_90d, users_90d,
engaged_sessions_90d, ai_sessions_90d, scroll_events_90d), and the Week-4 banned set
(trend_direction, trend_pct, all *_last_30d / *_prev_30d columns). Rows with avg_position == 0
are dropped, same as Week 4.

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

pd.set_option('display.width', 140)
np.random.seed(42)

# Download the dataset if it's not already present
!wget -q https://raw.githubusercontent.com/ghayoorahmed7/flyrank_ml_intership/main/data/content_refresh_anonymized.csv -P /content/

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df = df[df['avg_position'] > 0].copy()

banned = {
    'trend_direction', 'trend_pct',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'clicks_90d',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
}

target = 'ctr'
id_cols = {'content_id', 'client_id'}

candidate_features = [c for c in df.columns if c not in banned | id_cols | {target}]
print(f"{len(candidate_features)} candidate features (banned/id/target excluded):")
print(candidate_features)

23 candidate features (banned/id/target excluded):
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'avg_position', 'impression_tier', 'position_tier']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_id. My Week-4 notebook found that several top-ranked rows shared the same
client_id, even identical avg_position values — rows from the same client aren't independent.
A random row split would let the model see a client's other content in training and inflate
test performance. GroupShuffleSplit on client_id keeps each client entirely in train or test,
so the held-out score reflects generalization to unseen clients. I used an 80/20 grouped split,
plus grouped 5-fold CV to check the split wasn't a lucky draw.

In [9]:
X = df[candidate_features]
y = df[target]
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(f"train rows: {len(X_train)}, test rows: {len(X_test)}")
print(f"train clients: {groups_train.nunique()}, test clients: {groups.iloc[test_idx].nunique()}")
overlap = set(groups_train) & set(groups.iloc[test_idx])
print(f"client overlap between train/test: {len(overlap)} (should be 0)")

train rows: 22974, test rows: 5821
train clients: 24, test clients: 7
client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Baseline = tier-mean CTR, fit on the train split only, then applied to both splits by
position_tier — the same mechanic as Week 4's expected_ctr_by_tier, now held to proper
train/test discipline.

Metric: MAE on the ctr scale — directly interpretable as "how many CTR points off, on average,"
and robust to the outlier skew Week 4 already found in top-tier CTR. R² reported as a secondary
check.

In [10]:
tier_mean = X_train.join(y_train).groupby('position_tier')['ctr'].mean()
global_mean = y_train.mean()

baseline_train_pred = X_train['position_tier'].map(tier_mean).fillna(global_mean)
baseline_test_pred = X_test['position_tier'].map(tier_mean).fillna(global_mean)

baseline_train_mae = mean_absolute_error(y_train, baseline_train_pred)
baseline_test_mae = mean_absolute_error(y_test, baseline_test_pred)
baseline_test_r2 = r2_score(y_test, baseline_test_pred)

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

from sklearn.impute import SimpleImputer

# Define preprocessing for numerical features (imputation)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

# Define preprocessing for categorical features (imputation then one-hot encoding)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute before encoding
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)

models = {
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}

results = []
fitted_pipelines = {}
for name, reg in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', reg)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    pred_test = pipe.predict(X_test)
    results.append({
        'method': name,
        'test_MAE': mean_absolute_error(y_test, pred_test),
        'test_R2': r2_score(y_test, pred_test),
    })

results.append({'method': 'Baseline (tier-mean CTR)', 'test_MAE': baseline_test_mae, 'test_R2': baseline_test_r2})

comparison = pd.DataFrame(results).sort_values('test_MAE').reset_index(drop=True)
print(f"Baseline train MAE: {baseline_train_mae:.4f}")
print()
comparison

Baseline train MAE: 0.7854



,method,test_MAE,test_R2
0,Baseline (tier-mean CTR),0.693624,-0.209846
1,GradientBoosting,1.064098,-4.841199
2,RandomForest,1.693850,-9.346641


In [11]:
best_name = comparison.iloc[0]['method']
if best_name in fitted_pipelines:
    gkf = GroupKFold(n_splits=5)
    cv_scores = cross_val_score(
        fitted_pipelines[best_name], X, y, groups=groups,
        cv=gkf, scoring='neg_mean_absolute_error'
    )
    print(f"{best_name} grouped 5-fold CV MAE: {-cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
else:
    print("Best method was the baseline — no learned model to cross-validate.")

Best method was the baseline — no learned model to cross-validate.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
best_model_name = comparison[comparison['method'] != 'Baseline (tier-mean CTR)'].iloc[0]['method']
best_pipe = fitted_pipelines[best_model_name]
pred_test = best_pipe.predict(X_test)
resid = y_test.values - pred_test

err_df = X_test.copy()
err_df['ctr_actual'] = y_test.values
err_df['ctr_pred'] = pred_test
err_df['abs_error'] = np.abs(resid)

print("Largest errors, by position tier:")
print(err_df.groupby('position_tier')['abs_error'].agg(['mean', 'count']).round(3).sort_values('mean', ascending=False))
print()
print("Largest errors, by freshness tier:")
print(err_df.groupby('freshness_tier')['abs_error'].agg(['mean', 'count']).round(3).sort_values('mean', ascending=False))
print()
print("Worst 5 individual predictions:")
print(err_df.sort_values('abs_error', ascending=False)[['position_tier','freshness_tier','ctr_actual','ctr_pred','abs_error']].head(5))

Largest errors, by position tier:
                mean  count
position_tier              
top_3          6.112    318
deep           1.994    163
page_1         0.943   3052
page_3_5       0.459    963
striking       0.457   1325

Largest errors, by freshness tier:
                 mean  count
freshness_tier              
181+            5.002     21
0-30            1.137   4754
91-180          0.659   1036
31-90           0.228     10

Worst 5 individual predictions:
      position_tier freshness_tier  ctr_actual   ctr_pred  abs_error
7605          top_3           0-30         0.0  76.444061  76.444061
16827         top_3           0-30         0.0  73.904850  73.904850
7078          top_3           0-30         0.0  73.785358  73.785358
5992          top_3           0-30         0.0  73.339803  73.339803
6243          top_3           0-30         0.0  72.890425  72.890425


In [13]:
perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=10, random_state=42, scoring='neg_mean_absolute_error')
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

importance_df.head(10)

,feature,importance_mean,importance_std
18,word_count_tier,0.058665,0.002636
13,content_age_days,0.039769,0.007048
4,content_type,0.027565,0.013382
19,char_count_tier,0.024342,0.004679
7,char_count,0.016456,0.000929
6,word_count,0.016027,0.007272
12,days_with_sessions,0.009898,0.000603
3,cpc,0.000032,0.000016
21,impression_tier,0.000000,0.000000
2,competition_level,0.000000,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.